In [0]:
from pyspark.sql.functions import col, trim, dense_rank
from pyspark.sql.window import Window

# Läs in från Bronze layer i unity catalog
df_bronze = spark.read.table("marathos.bronze.raw_marathon")

# Filtrera lopp som har distans i km, mi elkler h samt gilltigt resultat
df_filtered = df_bronze.filter(
    col("Event distance/length").rlike("(?i)(km|mi|h)") &
    col("Athlete performance").isNotNull()
)

In [0]:
# Skapar window specifikationer
w_event = Window.orderBy("Event name")
w_athlete = Window.orderBy("Athlete ID")

# Genererar event_íd och athlete_íd med dense_rank()
df_silver = (
    df_filtered
    .withColumn("event_id", dense_rank().over(w_event))
    .withColumn("athlete_id", dense_rank().over(w_athlete))
    .withColumn("event_name_clean", trim(col("Event name")))

)

# Visar resultat
display(df_silver.select("event_id", "athlete_id", "Event name", "Event distance/length", "Athlete performance").limit(10))

In [0]:
# Sparar som delta tabell i marathos.silver.cleaned_marathon

(
    df_silver.write.format("delta")
    .option("delta.columnMapping.mode", "name")
    .mode("overwrite")
    .saveAsTable("marathos.silver.cleaned_marathon")
)

print("Silver layer har sparats i marathos.silver.cleaned_marathon!")